In [1]:
# -*- coding: utf-8 -*-
import pandas as pd
import re
import chardet
import os, time

# =========================
# 경로 설정
# =========================
INPUT_FILE = r"C:/Users/alstj/Documents/카카오톡 받은 파일/DB_recipe_steps_renamed.csv"
BASE_OUTPUT = r"C:/Users/alstj/Desktop/DB_recipe_steps_clean.csv"

# 열려있어도 저장되게 파일명 자동 변경
OUTPUT_FILE = BASE_OUTPUT if not os.path.exists(BASE_OUTPUT) else BASE_OUTPUT.replace(
    ".csv", f"_{time.strftime('%Y%m%d_%H%M%S')}.csv"
)

# =========================
# 인코딩 자동 감지
# =========================
def detect_encoding(path: str) -> str:
    with open(path, "rb") as f:
        raw = f.read()
    return chardet.detect(raw).get("encoding") or "utf-8"

# =========================
# 제거할 홍보/설명 키워드 패턴
# =========================
REMOVE_PATTERN = re.compile(
    r"(구독|좋아요|알림|댓글|더보기|설명란|링크|영상|다음\s*편|기대해|확인해\s*보|"
    r"채널|팔로우|공유|출처|문의|협찬|광고|커뮤니티)",
    re.IGNORECASE
)

def normalize_content(s: str) -> str:
    s = str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)      # 연속 공백 정리
    s = re.sub(r"\n{3,}", "\n\n", s)   # 과한 줄바꿈 정리
    return s.strip()

def should_remove(content: str) -> bool:
    if content is None:
        return True
    c = normalize_content(content)
    if c == "":
        return True
    # 홍보/설명 키워드 포함 시 제거
    if REMOVE_PATTERN.search(c):
        return True
    # 너무 짧은 감탄/엔딩 문장(선택적) - 원하면 끄기
    if len(c) <= 2:
        return True
    return False

# =========================
# 실행
# =========================
enc = detect_encoding(INPUT_FILE)
print("감지된 인코딩:", enc)

df = pd.read_csv(INPUT_FILE, encoding=enc)

# 필수 컬럼 확인(없으면 에러)
required_cols = {"recipe_id", "step_no", "content"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

# content 정리
df["content"] = df["content"].apply(normalize_content)

before = len(df)

# 제거
df_clean = df[~df["content"].apply(should_remove)].copy()

# step_no 정수화 & 정렬
df_clean["step_no"] = pd.to_numeric(df_clean["step_no"], errors="coerce")
df_clean = df_clean.dropna(subset=["step_no"])
df_clean["step_no"] = df_clean["step_no"].astype(int)

df_clean = df_clean.sort_values(["recipe_id", "step_no"]).reset_index(drop=True)

after = len(df_clean)

# 저장 (엑셀 안 깨지게)
df_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("✅ steps 정제 완료")
print("삭제된 행:", before - after)
print("남은 행:", after)
print("저장 위치:", OUTPUT_FILE)
print(df_clean.head(10))


감지된 인코딩: UTF-8-SIG
✅ steps 정제 완료
삭제된 행: 10
남은 행: 3975
저장 위치: C:/Users/alstj/Desktop/DB_recipe_steps_clean.csv
     recipe_id                                 title  step_no  \
0  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        1   
1  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        2   
2  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        3   
3  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        4   
4  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        5   
5  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        6   
6  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        7   
7  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        8   
8  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?        9   
9  -A0iP6Uzy3A  소떡소떡보다 맛있는 떡어묵꼬치! 어린이날 애들 간식으로 근사하죠?       10   

                                             content  
0                     떡은 흐르는 물에 세척한 후 물기를 제거하여 준비한다.  
1          *쌀 떡을 기름에 구울 경우 떡이 터질 수 있어 밀가루 떡을 사용해야한다.  
2        